# token-class

Assign a label to every token. This folder is a drop-in trainer for any token-classification dataset: swap the example in `preprocess.py` and `config.json`, then run.

## Example: emotion spans on GoEmotions

The bundled example finds emotion *spans* in Reddit comments from [GoEmotions](https://huggingface.co/datasets/google-research-datasets/go_emotions), using [sdeakin/GoEmotions-Projected-BIO-Emotions](https://huggingface.co/datasets/sdeakin/GoEmotions-Projected-BIO-Emotions). Each row is a comment plus character/token spans labeled with an emotion (`joy`, `sadness`, `anger`, …).

`preprocess.py` turns those spans into per-token BIO tags:

| token | tag |
| --- | --- |
| I | `O` |
| am | `O` |
| so | `B-Joy` |
| happy | `I-Joy` |
| today | `O` |

- `B-{emotion}` starts a span, `I-{emotion}` continues it, `O` is outside any span.
- Emotion names are taken from each span's `subtype` (or `type`) and normalized to `Joy`, `Sadness`, etc. Spans missing an emotion label or token indices are skipped.
- Empty trailing tokens are stripped after BIO tags are assigned (span indices refer to the original token list); comments with no tokens are dropped.
- Labels are collected from the data (`O` first, then every `B-*` / `I-*` tag that appears).
- The split is 90/10 train/val (`test_size` and `seed` in `config.json`).

Training fine-tunes `distilbert-base-uncased` as a token classifier. Wordpiece tokens inherit the word's label; subword continuations and special tokens are ignored (`-100`) so they do not affect loss or metrics. Eval runs each epoch; training stops if `span_f1` does not improve for `early_stopping_patience` epochs, and the best checkpoint is saved.

`evaluate.py` prints a short plain-English report on the held-out split:

- how often word tags match (this is high because most words are not part of an emotion)
- how many labeled emotion phrases the model got exactly right (same words and same emotion)
- precision, recall, and span F1 in everyday wording
- a few comments written as “labeled vs model,” with no token tables

`infer.py` prints the same kind of comment write-ups.

## Layout

```
.
├── config.json    # model, data URL, split, and training settings
├── preprocess.py  # load GoEmotions spans and build BIO labels
├── train.py       # train and save the best checkpoint
├── evaluate.py    # score the saved model in plain English
├── infer.py       # run the saved model on validation comments
├── README.md
```

Edit `config.json`, then from this folder:

```
python train.py
python evaluate.py
python infer.py
```


## Setup


In [ ]:
!pip install uv


In [ ]:
!uv pip install --system accelerate datasets numpy torch transformers


In [ ]:
from IPython.display import HTML, display
display(HTML("<style>pre,.output_text{white-space:pre-wrap!important;word-break:break-word!important}</style>"))


## Config


In [ ]:
%%writefile config.json
{
  "model": "distilbert-base-uncased",
  "max_length": 128,
  "seed": 42,
  "data_url": "https://huggingface.co/datasets/sdeakin/GoEmotions-Projected-BIO-Emotions/resolve/main/GoEmotions-Projected-BIO-Emotions.jsonl",
  "test_size": 0.1,
  "output_dir": "output",
  "num_train_epochs": 10,
  "per_device_train_batch_size": 16,
  "per_device_eval_batch_size": 32,
  "learning_rate": 5e-5,
  "weight_decay": 0.01,
  "warmup_ratio": 0.06,
  "eval_strategy": "epoch",
  "save_strategy": "epoch",
  "save_total_limit": 2,
  "logging_steps": 50,
  "metric_for_best_model": "span_f1",
  "early_stopping_patience": 2,
  "show_examples": 6
}


## Logs


In [ ]:
%%writefile progress.py
"""Keep tqdm/transformers logs readable when the output pane is narrow or resized.

Import this before `datasets` or `transformers` so progress bars are configured first.

Kaggle's log viewer does not treat ``\\r`` as overwrite, so a full-width tqdm bar
wraps into a staircase. This module turns those bars off in captured/Kaggle
output and uses a compact bar in a real terminal.
"""

from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path


def captured_display() -> bool:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return True
    if Path("/kaggle/input").exists() or Path("/kaggle/working").exists():
        return True
    if os.environ.get("COLAB_RELEASE_TAG"):
        return True
    try:
        return not sys.stdout.isatty()
    except Exception:
        return True


def disable_tqdm() -> bool:
    return captured_display()


def configure() -> None:
    warnings.filterwarnings("ignore", message="Was asked to gather along dimension 0", category=UserWarning)
    if disable_tqdm():
        os.environ["TQDM_DISABLE"] = "1"
        os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        return

    os.environ.setdefault("TQDM_DYNAMIC_NCOLS", "True")
    try:
        import tqdm.std as std
    except Exception:
        return

    original_init = std.tqdm.__init__

    def __init__(self, *args, **kwargs):
        kwargs.setdefault("dynamic_ncols", True)
        kwargs.setdefault("mininterval", 1.0)
        kwargs.setdefault(
            "bar_format",
            "{desc}: {percentage:3.0f}% {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )
        original_init(self, *args, **kwargs)

    std.tqdm.__init__ = __init__


configure()


## Preprocess


In [ ]:
%%writefile preprocess.py
"""
THIS FILE IS SPECIFICALLY FOR THE GOEMOTIONS BIO DATASET.

It loads sdeakin/GoEmotions-Projected-BIO-Emotions, converts emotion
spans into per-token BIO tags (B-Joy, I-Sadness, O, ...), and splits
train/val. Swap this file (and config.json) for any other
token-classification dataset.
"""

import json
import sys
from pathlib import Path

sys.path.append(str(Path(__file__).resolve().parents[1]))
import progress  # noqa: F401  # configure tqdm before datasets starts bars
from datasets import load_dataset

cfg = json.loads((Path(__file__).parent / "config.json").read_text())


def spans_to_bio(tokens, spans):
    tags = ["O"] * len(tokens)
    n = len(tokens)
    for span in spans:
        emotion = span.get("subtype") or span.get("type")
        start, end = span.get("start"), span.get("end")
        if not emotion or start is None or end is None or start < 0 or start >= n:
            continue
        emotion = emotion.replace(" ", "_").replace("-", "_")
        tags[start] = f"B-{emotion}"
        for i in range(start + 1, min(end, n - 1) + 1):
            tags[i] = f"I-{emotion}"
    return tags


def tag_spans(tags):
    spans, i = [], 0
    while i < len(tags):
        if tags[i].startswith("B-"):
            emotion, j = tags[i][2:], i + 1
            while j < len(tags) and tags[j] == f"I-{emotion}":
                j += 1
            spans.append((i, j, emotion))
            i = j
        else:
            i += 1
    return spans


def to_example(row):
    tokens = list(row["data"]["tokens"])
    tags = spans_to_bio(tokens, row["data"].get("spans") or [])
    while tokens and tokens[-1] == "":
        tokens.pop()
        tags.pop()
    return {
        "text": row["text"],
        "tokens": tokens,
        "bio_tags": tags,
    }


raw = load_dataset("json", data_files=cfg["data_url"], split="train")
ds = raw.map(to_example, remove_columns=raw.column_names)
ds = ds.filter(lambda row: len(row["tokens"]))
split = ds.train_test_split(test_size=cfg["test_size"], seed=cfg["seed"])
train_ds, eval_ds = split["train"], split["test"]

labels = sorted({tag for tags in ds["bio_tags"] for tag in tags})
labels.remove("O")
labels = ["O"] + labels
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for name, i in label2id.items()}


## Evaluate


In [ ]:
%%writefile evaluate.py
"""Score a trained token classifier. Example: emotion-span BIO tags."""

import json
import sys
from pathlib import Path

sys.path.append(str(Path(__file__).resolve().parents[1]))
import progress
import numpy as np
import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

from preprocess import eval_ds, id2label, label2id, tag_spans

root = Path(__file__).parent
cfg = json.loads((root / "config.json").read_text())


def make_tokenize(tokenizer):
    def tokenize(batch):
        encoded = tokenizer(
            batch["tokens"], is_split_into_words=True, truncation=True, max_length=cfg["max_length"]
        )
        aligned = []
        for i, tags in enumerate(batch["bio_tags"]):
            word_ids = encoded.word_ids(batch_index=i)
            ids, prev = [], None
            for word_id in word_ids:
                if word_id is None:
                    ids.append(-100)
                elif word_id != prev:
                    ids.append(label2id[tags[word_id]])
                else:
                    ids.append(-100)
                prev = word_id
            aligned.append(ids)
        encoded["labels"] = aligned
        return encoded

    return tokenize


def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    pred_ids = np.argmax(logits, axis=-1)
    gold_spans, pred_spans = [], []
    n_ok = n = 0
    offset = 0
    for pred_row, gold_row in zip(pred_ids, label_ids):
        gold, pred = [], []
        for p, g in zip(pred_row, gold_row):
            if g == -100:
                continue
            gold.append(id2label[int(g)])
            pred.append(id2label[int(p)])
            n_ok += int(p == g)
            n += 1
        gold_spans += [(s + offset, e + offset, emo) for s, e, emo in tag_spans(gold)]
        pred_spans += [(s + offset, e + offset, emo) for s, e, emo in tag_spans(pred)]
        offset += len(gold) + 1
    tp = len(set(gold_spans) & set(pred_spans))
    precision = tp / len(pred_spans) if pred_spans else 0.0
    recall = tp / len(gold_spans) if gold_spans else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "token_accuracy": n_ok / n,
        "span_precision": precision,
        "span_recall": recall,
        "span_f1": f1,
        "n_gold_spans": len(gold_spans),
        "n_pred_spans": len(pred_spans),
        "n_exact_span_hits": tp,
    }


def pct(x):
    return f"{100 * x:.1f}%"


def bio_to_spans(tokens, tags):
    return [(emo, " ".join(tokens[s:e])) for s, e, emo in tag_spans(tags)]


def describe_spans(spans):
    if not spans:
        return "no emotion"
    return "; ".join(f"{emo} in “{text}”" for emo, text in spans)


def describe_example(text, gold, pred):
    lines = [f"Comment: {text.strip()}"]
    if gold == pred:
        if gold:
            lines.append(f"Match: {describe_spans(gold)}.")
        else:
            lines.append("Match: neither the labels nor the model marked an emotion.")
        return "\n".join(lines)
    lines.append(f"Labeled: {describe_spans(gold)}.")
    lines.append(f"Model:   {describe_spans(pred)}.")
    return "\n".join(lines)


def print_metric_report(metrics):
    acc = metrics["eval_token_accuracy"]
    precision = metrics["eval_span_precision"]
    recall = metrics["eval_span_recall"]
    f1 = metrics["eval_span_f1"]
    n_gold = int(metrics["eval_n_gold_spans"])
    n_pred = int(metrics["eval_n_pred_spans"])
    n_hit = int(metrics["eval_n_exact_span_hits"])
    print("Held-out comments (validation split)")
    print(
        f"  Word tags: {pct(acc)} of words got the same BIO tag as the labels. "
        "Most words are ordinary (O), so this number runs high even when emotion phrases are shaky."
    )
    print(
        f"  Emotion phrases: the labels have {n_gold}, the model marked {n_pred}, "
        f"and {n_hit} match exactly (same words and same emotion)."
    )
    print(
        f"  When the model marks a phrase, it is exactly right {pct(precision)} of the time (precision)."
    )
    print(f"  It finds {pct(recall)} of the labeled phrases (recall).")
    print(
        f"  Combined score (span F1): {pct(f1)}. "
        "A hit has to get both the phrase boundaries and the emotion name right."
    )


def print_examples(pred_ids, label_ids, n):
    print("\nA few comments, in plain English")
    shown = 0
    for row, pred_row, gold_row in zip(eval_ds, pred_ids, label_ids):
        pred = [id2label[int(p)] for p, g in zip(pred_row, gold_row) if g != -100]
        if len(pred) != len(row["tokens"]):
            continue
        gold_spans = bio_to_spans(row["tokens"], row["bio_tags"])
        pred_spans = bio_to_spans(row["tokens"], pred)
        if not gold_spans and not pred_spans:
            continue
        print()
        print(describe_example(row["text"], gold_spans, pred_spans))
        shown += 1
        if shown == n:
            break


def predict_tags(model, tokenizer, tokens):
    tags = ["O"] * len(tokens)
    device = next(model.parameters()).device
    start = 0
    while start < len(tokens):
        encoded = tokenizer(
            tokens[start:],
            is_split_into_words=True,
            truncation=True,
            max_length=cfg["max_length"],
            return_tensors="pt",
        )
        word_ids = encoded.word_ids()
        inputs = {k: encoded[k].to(device) for k in ("input_ids", "attention_mask")}
        with torch.no_grad():
            pred_ids = model(**inputs).logits[0].argmax(-1).tolist()
        seen = set()
        last = -1
        for word_id, pred_id in zip(word_ids, pred_ids):
            if word_id is None or word_id in seen:
                continue
            tags[start + word_id] = id2label[pred_id]
            seen.add(word_id)
            last = word_id
        if last < 0:
            break
        start += last + 1
    return tags


if __name__ == "__main__":
    model_dir = str(root / cfg["output_dir"] / "best_model")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForTokenClassification.from_pretrained(model_dir)
    tokenized_eval = eval_ds.map(
        make_tokenize(tokenizer), batched=True, remove_columns=eval_ds.column_names
    )
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=str(root / cfg["output_dir"]),
            per_device_eval_batch_size=cfg["per_device_eval_batch_size"],
            fp16=torch.cuda.is_available(),
            report_to="none",
            disable_tqdm=progress.disable_tqdm(),
        ),
        eval_dataset=tokenized_eval,
        processing_class=tokenizer,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )
    output = trainer.predict(tokenized_eval, metric_key_prefix="eval")
    print_metric_report(output.metrics)
    print_examples(np.argmax(output.predictions, axis=-1), output.label_ids, cfg["show_examples"])


## Train


In [ ]:
%%writefile train.py
"""Train a token classifier. Example labels: emotion-span BIO tags."""

import json
import os
import sys
from pathlib import Path

sys.path.append(str(Path(__file__).resolve().parents[1]))
import progress
import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

from evaluate import compute_metrics, make_tokenize
from preprocess import eval_ds, id2label, label2id, labels, train_ds

root = Path(__file__).parent
cfg = json.loads((root / "config.json").read_text())
output_dir = str(root / cfg["output_dir"])

os.environ["WANDB_DISABLED"] = "true"
set_seed(cfg["seed"])

print(len(train_ds), "train,", len(eval_ds), "val,", len(labels), "labels")

tokenizer = AutoTokenizer.from_pretrained(cfg["model"])
model = AutoModelForTokenClassification.from_pretrained(
    cfg["model"], num_labels=len(labels), id2label=id2label, label2id=label2id
)

tokenize = make_tokenize(tokenizer)
tokenized_train = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
tokenized_eval = eval_ds.map(tokenize, batched=True, remove_columns=eval_ds.column_names)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=cfg["num_train_epochs"],
        per_device_train_batch_size=cfg["per_device_train_batch_size"],
        per_device_eval_batch_size=cfg["per_device_eval_batch_size"],
        learning_rate=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
        warmup_ratio=cfg["warmup_ratio"],
        eval_strategy=cfg["eval_strategy"],
        save_strategy=cfg["save_strategy"],
        save_total_limit=cfg["save_total_limit"],
        load_best_model_at_end=True,
        metric_for_best_model=cfg["metric_for_best_model"],
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=cfg["seed"],
        disable_tqdm=progress.disable_tqdm(),
        logging_steps=cfg["logging_steps"],
        logging_first_step=True,
    ),
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["early_stopping_patience"])],
)
trainer.train()
trainer.save_model(str(Path(output_dir) / "best_model"))


In [ ]:
!python train.py


In [ ]:
!python evaluate.py
